# Phase 1 - Ingest + Normalize

Goal: pull a 2M stratified sample of NYC 311 records, normalize column names across the 2020+ and 2010-19 datasets, write to a single parquet on Drive.

Phase 0 must already be passing before you start this. If it isnt, the spark session and dependencies wont be ready.

## Cell 1 - Bootstrap (drive + repo + spark)

This is the standard cold-start cell - same pattern in every phase notebook so we can re-run any notebook on a fresh kernel without manual setup.

In [ ]:
# project repo
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

# mount drive
from google.colab import drive
drive.mount('/content/drive')

# pull latest from repo
import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

# install deps if not already present (cheap if already installed)
!pip install -r /content/project/requirements-train.txt -q

# java 11
!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# soda token if available
try:
    from google.colab import userdata
    tok = userdata.get('SODA_APP_TOKEN')
    if tok:
        os.environ['SODA_APP_TOKEN'] = tok
except Exception:
    pass

# spark up
from src.spark_setup import get_spark
spark = get_spark(app_name='phase1-ingest')
print('spark', spark.version, 'ready')

## Cell 2 - Pull 1M rows from the 2020+ dataset

Stable order by `unique_key` so pagination is deterministic. With a SODA app token, ~1M rows takes 15-25 minutes.

In [ ]:
from src.config import SODA_2020_PLUS, SODA_2010_2019
from src.ingest import fetch_311_to_parquet

# write directly to drive so re-running this cell is idempotent
out_path_2020 = '/content/drive/MyDrive/cs6513/raw/2020plus.parquet'
n_2020 = fetch_311_to_parquet(
    spark=spark,
    endpoint=SODA_2020_PLUS,
    target_rows=1_000_000,
    out_path=out_path_2020,
)
print(f'\n2020+ done. {n_2020:,} rows on disk at {out_path_2020}')

## Cell 3 - Pull 1M rows from the 2010-2019 dataset

In [ ]:
out_path_hist = '/content/drive/MyDrive/cs6513/raw/historical.parquet'
n_hist = fetch_311_to_parquet(
    spark=spark,
    endpoint=SODA_2010_2019,
    target_rows=1_000_000,
    out_path=out_path_hist,
)
print(f'\n2010-2019 done. {n_hist:,} rows on disk at {out_path_hist}')

## Cell 4 - Union both halves with normalized schema

In [ ]:
# read both back, union, save the combined parquet
df_2020 = spark.read.parquet(out_path_2020)
df_hist = spark.read.parquet(out_path_hist)

# both should already be normalized from fetch_311_to_parquet but lets be sure
from src.ingest import normalize_columns
df_2020 = normalize_columns(df_2020)
df_hist = normalize_columns(df_hist)

combined = df_2020.unionByName(df_hist, allowMissingColumns=True)
print(f'combined row count: {combined.count():,}')
combined.printSchema()

## Cell 5 - Stratified 2M sample

We use `sampleBy` so the per-class proportions match the full corpus. Random split would underrepresent minority categories.

In [ ]:
from src.ingest import stratified_sample
from src.config import SAMPLE_SIZE

sample = stratified_sample(combined, target_size=SAMPLE_SIZE, label_col='problem')
actual_count = sample.count()
print(f'stratified sample size: {actual_count:,}')

# write to drive. partition by year extracted from created_date so later
# notebooks can filter by year cheaply.
from pyspark.sql import functions as F
sample = sample.withColumn('year', F.year('created_date'))
out_sample = '/content/drive/MyDrive/cs6513/sample_2m.parquet'
sample.write.mode('overwrite').partitionBy('year').parquet(out_sample)
print(f'sample written to {out_sample}')

## Cell 6 - Class distribution histogram

Sanity check: top 20 categories should match the well-known NYC 311 distribution (Noise, Heat/Hot Water, Illegal Parking dominate).

In [ ]:
import matplotlib.pyplot as plt

top20 = sample.groupBy('problem').count().orderBy(F.desc('count')).limit(20).toPandas()
print(top20)

# save plot to dashboard assets so the streamlit pipeline status tab can show it
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20['problem'][::-1], top20['count'][::-1])
ax.set_xlabel('count')
ax.set_title('top 20 complaint categories in 2m stratified sample')
plt.tight_layout()
plt.savefig('/content/project/dashboard/assets/class_dist.png', dpi=120)
plt.show()

print('saved plot to dashboard/assets/class_dist.png')

## Phase 1 - Done when

- `sample_2m.parquet` on Drive with row count exactly 2,000,000 (Cell 5).
- Top-20 histogram looks like a typical 311 distribution (Cell 6 plot).
- Schema printed in Cell 4 includes all KEEP_COLS from `src/ingest.py`.

Then commit the saved notebook + the new `class_dist.png` to GitHub and move to `02_preprocess.ipynb` for Phase 2.